In [1]:
import pandas as pd
import numpy as np

In [2]:
import os
os.getcwd()


'/Users/himanirohaj/Desktop/ds_project/app'

In [3]:
df = pd.read_json("../data/problems.json", lines=True)

df.head()


,title,description,input_description,output_description,sample_io,problem_class,problem_score,url
0,Uuu,Unununium (Uuu) was the name of the chemical\n...,The input consists of one line with two intege...,The output consists of $M$ lines where the $i$...,"[{'input': '7 10', 'output': '1 2 2 3 1 3 3 4 ...",hard,9.7,https://open.kattis.com/problems/uuu
1,House Building,A number of eccentrics from central New York h...,"The input consists of $10$ test cases, which a...",Print $K$ lines with\n the positions of the...,"[{'input': '0 2 3 2 50 60 50 30 50 40', 'outpu...",hard,9.7,https://open.kattis.com/problems/husbygge
2,Mario or Luigi,Mario and Luigi are playing a game where they ...,,,"[{'input': '', 'output': ''}]",hard,9.6,https://open.kattis.com/problems/marioorluigi
3,The Wire Ghost,Žofka is bending a copper wire. She starts wit...,The first line contains two integers $L$ and $...,The output consists of a single line consistin...,"[{'input': '4 3 3 C 2 C 1 C', 'output': 'GHOST...",hard,9.6,https://open.kattis.com/problems/thewireghost
4,Barking Up The Wrong Tree,"Your dog Spot is let loose in the park. Well, ...",The first line of input consists of two intege...,Write a single line containing the length need...,"[{'input': '2 0 10 0 10 10', 'output': '14.14'...",hard,9.6,https://open.kattis.com/problems/barktree


In [4]:
df.columns


Index(['title', 'description', 'input_description', 'output_description',
       'sample_io', 'problem_class', 'problem_score', 'url'],
      dtype='object')

In [5]:
df = df.fillna("")

df["combined_text"] = (
    df["title"] + " " +
    df["description"] + " " +
    df["input_description"] + " " +
    df["output_description"]
)

X = df["combined_text"]
y_class = df["problem_class"]
y_score = df["problem_score"]


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer


In [7]:
tfidf = TfidfVectorizer(
    max_features=8000,
    ngram_range=(1, 2),
    stop_words="english"
)

In [8]:
X_tfidf = tfidf.fit_transform(X)


In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_class_train, y_class_test = train_test_split(
    X_tfidf, y_class, test_size=0.2, random_state=42
)

_, _, y_score_train, y_score_test = train_test_split(
    X_tfidf, y_score, test_size=0.2, random_state=42
)

In [10]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

clf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    random_state=42
)

clf.fit(X_train, y_class_train)

y_class_pred = clf.predict(X_test)

accuracy_score(y_class_test, y_class_pred)


0.5540704738760632

In [11]:
from sklearn.metrics import confusion_matrix


In [12]:
confusion_matrix(y_class_test, y_class_pred)


array([[ 30,  68,  38],
       [  7, 378,  40],
       [ 14, 200,  48]])

In [13]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

reg = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

reg.fit(X_train, y_score_train)

y_score_pred = reg.predict(X_test)

mean_absolute_error(y_score_test, y_score_pred), np.sqrt(mean_squared_error(y_score_test, y_score_pred))


(1.7048718104495748, np.float64(2.0403685952681445))

In [14]:
import joblib

joblib.dump(tfidf, "tfidf.pkl")
joblib.dump(clf, "classifier.pkl")
joblib.dump(reg, "regressor.pkl")


['regressor.pkl']

In [15]:
df["problem_score"].describe()


count    4112.000000
mean        5.114689
std         2.177770
min         1.100000
25%         3.300000
50%         5.200000
75%         6.900000
max         9.700000
Name: problem_score, dtype: float64